# MasakApa AI - Data Exploration & Model Prototyping

Di dalam Content-Based Filtering menggunakan TF-IDF, tidak ada proses "training" (seperti backpropagation di Neural Network). Proses "training" di sini adalah **fitting** teks (bahan masakan) ke dalam ruang vektor untuk mendapatkan nilai Term Frequency-Inverse Document Frequency.

Notebook ini dibuat untuk mendemonstrasikan:
1. Eksplorasi Data (EDA)
2. Preprocessing Teks
3. Pembuatan Model (TF-IDF)
4. Evaluasi (Cosine Similarity)

*(Catatan: Di dalam API backend, model TF-IDF dilatih secara on-the-fly ketika server dinyalakan karena prosesnya sangat cepat. Namun notebook ini penting untuk dokumentasi Capstone Project.)*

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

# Setting untuk mempermudah melihat data pandas
pd.set_option('display.max_colwidth', None)

## 1. Load Dataset
Kita memuat dataset resep masakan Indonesia.

In [2]:
df = pd.read_csv('../data/Indonesian_Food_Recipes.csv')
print(f"Total Data: {df.shape[0]} baris dan {df.shape[1]} kolom.")
df[['Title', 'Ingredients Cleaned', 'Category']].head()

Total Data: 14945 baris dan 10 kolom.


,Title,Ingredients Cleaned,Category
0,Ayam Woku Manado,"ayam kampung potong , jeruk nipis , garam , kunyit , bawang merah , bawang putih , cabe merah , cabe rawit merah , kemiri , serai , daun salam , daun kemangi , penyedap , air",ayam
1,Ayam goreng tulang lunak,"ayam dipotong , serai , daun jeruk , bawang putih haluskan , biji ketumbar , laos haluskan , kunyit , kemiri , garam , air tuk ukep ayam , minyak goreng",ayam
2,Ayam cabai kawin,"ayam , cabai hijau , cabai merah rawit , bawang putih , bawang merah , gula , garam , tomat merah , air , minyak goreng",ayam
3,Ayam Geprek,"daging ayam fillet , gula garam , tepung ayam , daun kemangi , kol , timun , minyak panas , sambal korek , cabe rawit merah bawang putih",ayam
4,Minyak Ayam,"kulit ayam & lemaknya , bawang putih , cincang kasar , jahe , geprek , minyak goreng , biji ketumbar",ayam


## 2. Preprocessing Data
Kita membersihkan kolom `Ingredients Cleaned` dan menyatukan sinonim (misal: cabai merah -> cabai).

In [3]:
SYNONYMS = {
    "cabai merah": "cabai",
    "cabe": "cabai",
    "bawang merah": "bawang",
    "bawang putih": "bawang putih",
    "ayam broiler": "ayam",
    "daging ayam": "ayam",
}

def preprocess_ingredients(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    for k, v in SYNONYMS.items():
        text = text.replace(k, v)
    
    # Ekstrak per bahan (dipisahkan koma)
    items = text.split(',')
    processed = []
    for item in items:
        item = re.sub(r'[^\w\s]', ' ', item)
        item = " ".join(item.split())
        if item:
            processed.append(item)
            
    return " ".join(processed)

df['Ingredients Cleaned'] = df['Ingredients Cleaned'].fillna("")
df['Processed_Ingredients'] = df['Ingredients Cleaned'].apply(preprocess_ingredients)
df[['Ingredients Cleaned', 'Processed_Ingredients']].head()

,Ingredients Cleaned,Processed_Ingredients
0,"ayam kampung potong , jeruk nipis , garam , kunyit , bawang merah , bawang putih , cabe merah , cabe rawit merah , kemiri , serai , daun salam , daun kemangi , penyedap , air",ayam kampung potong jeruk nipis garam kunyit bawang bawang putih cabai merah cabai rawit merah kemiri serai daun salam daun kemangi penyedap air
1,"ayam dipotong , serai , daun jeruk , bawang putih haluskan , biji ketumbar , laos haluskan , kunyit , kemiri , garam , air tuk ukep ayam , minyak goreng",ayam dipotong serai daun jeruk bawang putih haluskan biji ketumbar laos haluskan kunyit kemiri garam air tuk ukep ayam minyak goreng
2,"ayam , cabai hijau , cabai merah rawit , bawang putih , bawang merah , gula , garam , tomat merah , air , minyak goreng",ayam cabai hijau cabai rawit bawang putih bawang gula garam tomat merah air minyak goreng
3,"daging ayam fillet , gula garam , tepung ayam , daun kemangi , kol , timun , minyak panas , sambal korek , cabe rawit merah bawang putih",ayam fillet gula garam tepung ayam daun kemangi kol timun minyak panas sambal korek cabai rawit merah bawang putih
4,"kulit ayam & lemaknya , bawang putih , cincang kasar , jahe , geprek , minyak goreng , biji ketumbar",kulit ayam lemaknya bawang putih cincang kasar jahe geprek minyak goreng biji ketumbar


## 3. Modeling dengan TF-IDF
Melatih (`fit`) TfidfVectorizer terhadap teks bahan masakan agar kata-kata diubah menjadi vektor representasi matriks.

In [4]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['Processed_Ingredients'])

print("Ukuran TF-IDF Matrix:", tfidf_matrix.shape)
print("Fitur pertama (contoh kata):", vectorizer.get_feature_names_out()[:10])

Ukuran TF-IDF Matrix: (14945, 6571)
Fitur pertama (contoh kata): ['___________________' '____________________________' '_potong' 'aa' 'ab'
 'abaikan' 'abang' 'abangnya' 'abc' 'abis']


## 4. Evaluasi Rekomendasi dengan Cosine Similarity
Menguji input pengguna, mengubahnya ke TF-IDF, dan mencari resep dengan Cosine Similarity teratas.

In [5]:
user_input = "ayam, bawang putih, cabai"
processed_input = preprocess_ingredients(user_input)

user_vector = vectorizer.transform([processed_input])
similarities = cosine_similarity(user_vector, tfidf_matrix).flatten()

# Tambahkan skor ke dataframe
df['similarity_score'] = similarities

# Ambil top 5
top_5 = df.sort_values(by=['similarity_score', 'Loves'], ascending=[False, False]).head(5)
top_5[['Title', 'Category', 'similarity_score', 'Loves']]

,Title,Category,similarity_score,Loves
10862,Telur Sambel Korek,telur,0.653669,5
10098,Telur Balado,telur,0.641249,20
674,2. Ayam Geprek,ayam,0.623785,8
10268,Telur balado,telur,0.611547,3
10583,Telur saus sambal,telur,0.606480,1


## 5. Export Model untuk Backend MVP
Menyimpan model (TF-IDF Vectorizer & Matrix) beserta dataset yang sudah diproses ke dalam folder `models/` agar bisa di-load oleh backend FastAPI tanpa harus di-training ulang.

In [6]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

# Simpan Vectorizer
joblib.dump(vectorizer, '../models/vectorizer.pkl')

# Simpan Matrix TF-IDF
joblib.dump(tfidf_matrix, '../models/tfidf_matrix.pkl')

# Simpan DataFrame yang sudah dibersihkan
df.to_pickle('../models/df_recipes.pkl')

print("Model berhasil diexport ke folder models/")

Model berhasil diexport ke folder models/
